## 1. CHARGEMENT DES DONNÉES
On importe les deux fichiers Excel dans Python

In [10]:
import pandas as pd

# Configuration de l'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1100)
pd.set_option('display.max_colwidth', 15)

# Chargement des commandes
df_commandes = pd.read_excel(
    "../data/brut/commande_produit_blanc_2017_2025.xlsx"
)

df_stations = pd.read_excel(
    "../data/brut/liste des station.xlsx"
)

# Aperçu des données
print("=== COMMANDES ===")
print(df_commandes.shape)
display(df_commandes.head())

print("\n=== STATIONS ===")
print(df_stations.shape)
display(df_stations.head())

=== COMMANDES ===
(51853, 13)


,Date comptabilisation,N° document,N° article,N° origine,N° contact origine,Quantité facturée,Montant vente (réel),Coût total (réel),Code Produit,Type écriture,Type document,Code magasin,N° commande
0,2017-02-03,BL170352,A0001,C0437,NaN,-4000.0,1591724.0,-1420103.0,PROD-BLANC,Vente,Expédition ...,ORYX,NaN
1,2017-02-03,BL170352,A0002,C0437,NaN,-4000.0,1568818.0,0.0,PROD-BLANC,Vente,Expédition ...,ORYX,NaN
2,2017-02-03,BL170353,A0001,C0001,NaN,-3000.0,1202793.0,-1065077.0,PROD-BLANC,Vente,Expédition ...,ORYX,NaN
3,2017-02-03,BL170353,A0002,C0001,NaN,-30000.0,11856138.0,0.0,PROD-BLANC,Vente,Expédition ...,ORYX,NaN
4,2017-02-03,BL170354,A0001,C0387,NaN,-4000.0,1591724.0,-1420103.0,PROD-BLANC,Vente,Expédition ...,ORYX,NaN



=== STATIONS ===
(147, 8)


,Code Contact Client,NOM CLIENT,Code_sous_compte_Client,Nom sous compte,TYPE,STATUT,Code Station,GROUPE TYPE
0,C0083,2M ASSOCIES,CC6471,STATION BOU...,CODO,Fonctionnelle,ST0001,STATION
1,C0083,2M ASSOCIES,CC3018,STATION COM...,CODO,Fonctionnelle,ST0002,STATION
2,C0438,ABBE SOLO,CC3045,STATION HEVIE,CODO,Ancien gérant,ST0003,STATION
3,C2472,ADOUNI PLUS,CC6791,STATION SUR...,CODO,Fonctionnelle,ST0004,STATION
4,C2241,ADOUNI PLUS...,CC6139,STATION SUR...,CODO,Ancien gérant,ST0004,STATION


Le fichier commandes contient **51 853 lignes et le fichier stations 147 stations**.

## 2. FUSION DES DEUX FICHIERS
On relie les commandes aux stations via le code client. Colonne clé : "N° origine" (commandes) = "Code Contact Client" (stations)

Pourquoi cette étape est importante

Cette fusion permet :

d’unifier les stations,
récupérer le vrai identifiant station,
préparer les données pour l’analyse temporelle.

C’est une étape clé du projet.

In [15]:
# Fusion des commandes avec le référentiel stations
df = pd.merge(
    df_commandes,
    df_stations,
    left_on="N° contact origine",
    right_on="Code_sous_compte_Client",
    how="left"
)

# Vérification
print("=== DIMENSIONS APRES FUSION ===")
print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])

# Aperçu
display(
    df[
        [
            "Date comptabilisation",
            "N° contact origine",
            "NOM CLIENT",
            "Code Station",
            "Quantité facturée"
        ]
    ].head()
)

=== DIMENSIONS APRES FUSION ===
Nombre de lignes : 51853
Nombre de colonnes : 21


,Date comptabilisation,N° contact origine,NOM CLIENT,Code Station,Quantité facturée
0,2017-02-03,NaN,NaN,NaN,-4000.0
1,2017-02-03,NaN,NaN,NaN,-4000.0
2,2017-02-03,NaN,NaN,NaN,-3000.0
3,2017-02-03,NaN,NaN,NaN,-30000.0
4,2017-02-03,NaN,NaN,NaN,-4000.0


## 3. NETTOYAGE DES DONNÉES
Avant toute analyse, on vérifie et corrige la qualité des données

Le Machine Learning nécessite :

des données propres,
cohérentes,
sans valeurs erronées.

Cette étape évite :

les biais,
les erreurs de prédiction,
les problèmes d’apprentissage.

In [19]:
# --- Conversion des dates ---
df["Date comptabilisation"] = pd.to_datetime(
    df["Date comptabilisation"]
)

# --- Vérification des valeurs manquantes ---
print("=== VALEURS MANQUANTES ===")
print(df.isnull().sum())

# --- Suppression des lignes sans station identifiée ---
df_clean = df.dropna(
    subset=["Code Station"]
).copy()

print("\n=== NETTOYAGE STATIONS ===")
print("Avant nettoyage :", df.shape[0])
print("Après nettoyage :", df_clean.shape[0])
print("Lignes supprimées :", df.shape[0] - df_clean.shape[0])

# --- Correction des quantités négatives ---
df_clean["Quantité facturée"] = (
    df_clean["Quantité facturée"].abs()
)

print("\n=== STATISTIQUES QUANTITES ===")
display(
    df_clean["Quantité facturée"].describe()
)

# --- Suppression des colonnes inutiles ---
colonnes_a_supprimer = [
    "N° document",
    "Montant vente (réel)",
    "Coût total (réel)",
    "Type écriture",
    "Code magasin",
    "Code_sous_compte_Client",
    "Code Contact Client",
    "N° commande"
]

df_clean = df_clean.drop(
    columns=colonnes_a_supprimer,
    errors="ignore"
)

=== VALEURS MANQUANTES ===
Date comptabilisation          0
N° document                    0
N° article                     0
N° origine                     0
N° contact origine         12169
Quantité facturée              0
Montant vente (réel)           0
Coût total (réel)              0
Code Produit                   0
Type écriture                  0
Type document                  0
Code magasin                   0
N° commande                 8465
Code Contact Client        17703
NOM CLIENT                 17703
Code_sous_compte_Client    17703
Nom sous compte            17703
TYPE                       17703
STATUT                     17703
Code Station               17703
GROUPE TYPE                17703
dtype: int64

=== NETTOYAGE STATIONS ===
Avant nettoyage : 51853
Après nettoyage : 34150
Lignes supprimées : 17703

=== STATISTIQUES QUANTITES ===


count    34150.000000
mean      7977.507116
std       6100.131202
min          8.000000
25%       4000.000000
50%       6000.000000
75%      10000.000000
max      48000.000000
Name: Quantité facturée, dtype: float64

## 4. VERIFICATION FINALE
Elle permet :

de figer une version propre des données,
éviter de refaire le preprocessing,
alimenter directement les prochaines étapes :
feature engineering,
ML,
forecasting.

In [24]:
print("=== COLONNES FINALES ===")

for col in df_clean.columns:
    print("-", col)

print("\n=== DIMENSIONS FINALES ===")
print(df_clean.shape)

# Aperçu final
display(df_clean.head())

# Sauvegarde du dataset propre
df_clean.to_excel(
    "../data/propre/commandes_propres.xlsx",
    index=False
)

print("\nDataset sauvegardé avec succès.")

=== COLONNES FINALES ===
- Date comptabilisation
- N° article
- N° origine
- N° contact origine
- Quantité facturée
- Code Produit
- Type document
- NOM CLIENT
- Nom sous compte
- TYPE
- STATUT
- Code Station
- GROUPE TYPE

=== DIMENSIONS FINALES ===
(34150, 13)


,Date comptabilisation,N° article,N° origine,N° contact origine,Quantité facturée,Code Produit,Type document,NOM CLIENT,Nom sous compte,TYPE,STATUT,Code Station,GROUPE TYPE
12171,2020-01-13,A0001,C1234,CC2616,5000.0,PROD-BLANC,Expédition ...,GIBLAND COM...,STATION ATTAKE,CODO,Fonctionnelle,ST0049,STATION
12172,2020-01-13,A0002,C1234,CC2616,4000.0,PROD-BLANC,Expédition ...,GIBLAND COM...,STATION ATTAKE,CODO,Fonctionnelle,ST0049,STATION
12173,2020-01-13,A0001,C1249,CC2650,1000.0,PROD-BLANC,Expédition ...,TONI ET FILS,STATION NDALI,DODO,Fonctionnelle,ST0092,STATION
12174,2020-01-13,A0002,C1249,CC2650,16000.0,PROD-BLANC,Expédition ...,TONI ET FILS,STATION NDALI,DODO,Fonctionnelle,ST0092,STATION
12178,2020-01-13,A0001,C1233,CC2614,5000.0,PROD-BLANC,Expédition ...,AYODELE BUS...,STATION OUIDAH,CODO,Fonctionnelle,ST0016,STATION



Dataset sauvegardé avec succès.
